In [1]:
print("Hello")

Hello


In [2]:
from pathlib import Path
import open3d as o3d
import cv2
import matplotlib.pyplot as plt
import numpy as np
import glob as gb
import os



def paired_names(folder):
    left_names = {path.name for path in (folder / "left").glob("*.png")}
    right_names = {path.name for path in (folder / "right").glob("*.png")}
    return sorted(left_names & right_names)

def read_pair(folder, name):
    left = cv2.imread(str(folder / "left" / name))
    right = cv2.imread(str(folder / "right" / name))
    return left, right

def rectify_pair(left, right, calibration):
    # Keep the full image so the pixel coordinates still match P1, P2 and Q.
    rectified_left = cv2.remap(left, calibration["map_left_x"], calibration["map_left_y"], cv2.INTER_LINEAR)
    rectified_right = cv2.remap(right, calibration["map_right_x"], calibration["map_right_y"], cv2.INTER_LINEAR)
    return rectified_left, rectified_right


def calibration(project_dir, save_calib=False ):
    project_dir = Path(project_dir)
    calibration_dir = project_dir / "0_calib"
    calibration_names = paired_names(calibration_dir)
    cols, rows = 7, 5
    pattern_size = (cols, rows)
    square_size = 0.08  # One checkerboard square, 0.08 meter or 80mm.

    objp = np.zeros((cols * rows, 3), np.float32)
    objp[:, :2] = np.mgrid[:cols, :rows].T.reshape(-1, 2) * square_size

    objpoints = []
    imgpoints_left = []
    imgpoints_right = []
    reversed_pairs = []
    # This detector already returns subpixel corner positions.
    corner_flags = cv2.CALIB_CB_EXHAUSTIVE | cv2.CALIB_CB_ACCURACY

    for name in calibration_names:
        left, right = read_pair(calibration_dir, name)
        gray_left = cv2.cvtColor(left, cv2.COLOR_BGR2GRAY)
        gray_right = cv2.cvtColor(right, cv2.COLOR_BGR2GRAY)
        found_left, corners_left = cv2.findChessboardCornersSB(gray_left, pattern_size, flags=corner_flags)
        found_right, corners_right = cv2.findChessboardCornersSB(gray_right, pattern_size, flags=corner_flags)
        if not (found_left and found_right):
            continue

        corners_left = corners_left.reshape(-1, 1, 2)
        corners_right = corners_right.reshape(-1, 1, 2)
        # A symmetric board can be detected starting from opposite ends.
        left_direction = corners_left[cols - 1, 0] - corners_left[0, 0]
        right_direction = corners_right[cols - 1, 0] - corners_right[0, 0]
        if np.dot(left_direction, right_direction) < 0:
            corners_right = corners_right[::-1].copy()
            reversed_pairs.append(name)

        objpoints.append(objp)
        imgpoints_left.append(corners_left)
        imgpoints_right.append(corners_right)

    assert len(objpoints) >= 3, "Too few stereo pairs with checkerboard corners."
    image_size = gray_left.shape[::-1]
    print(f"Usable stereo pairs: {len(objpoints)}/{len(calibration_names)}")
    print(f"Right-side corner order reversed in {len(reversed_pairs)} pairs.")

    rms_left, mtx_left, dist_left, _, _ = cv2.calibrateCamera(
        objpoints, imgpoints_left, image_size, None, None
    )
    rms_right, mtx_right, dist_right, _, _ = cv2.calibrateCamera(
        objpoints, imgpoints_right, image_size, None, None
    )
    rms_stereo, mtx_left, dist_left, mtx_right, dist_right, R, T, _, _ = cv2.stereoCalibrate(
        objpoints, imgpoints_left, imgpoints_right,
        mtx_left, dist_left, mtx_right, dist_right, image_size,
        flags=cv2.CALIB_FIX_INTRINSIC,
    )

    R1, R2, P1, P2, Q, _, _ = cv2.stereoRectify(
        mtx_left, dist_left, mtx_right, dist_right, image_size, R, T,
        flags=cv2.CALIB_ZERO_DISPARITY, alpha=0,
    )

    rectified_corners_left = cv2.undistortPoints(
        np.concatenate(imgpoints_left), mtx_left, dist_left, R=R1, P=P1
    ).reshape(-1, 2)
    rectified_corners_right = cv2.undistortPoints(
        np.concatenate(imgpoints_right), mtx_right, dist_right, R=R2, P=P2
    ).reshape(-1, 2)
    row_error = np.abs(rectified_corners_left[:, 1] - rectified_corners_right[:, 1])

    print(f"Reprojection RMS: left {rms_left:.2f}, right {rms_right:.2f}, stereo {rms_stereo:.2f} px")
    print(f"Rectified row difference: median {np.median(row_error):.2f} px, "
          f"95th percentile {np.percentile(row_error, 95):.2f} px")
    if save_calib:
        np.savez(
            calibration_dir / "stereo_calibration.npz",
            mtx_left=mtx_left, dist_left=dist_left,
            mtx_right=mtx_right, dist_right=dist_right,
            R=R, T=T, R1=R1, R2=R2, P1=P1, P2=P2, Q=Q,
            image_size=image_size, square_size=square_size,
        )
    map_left_x, map_left_y = cv2.initUndistortRectifyMap(
        mtx_left, dist_left, R1, P1, image_size, cv2.CV_32FC1
    )
    map_right_x, map_right_y = cv2.initUndistortRectifyMap(
        mtx_right, dist_right, R2, P2, image_size, cv2.CV_32FC1
    )
    calibration = {
        "mtx_left": mtx_left, "dist_left": dist_left,
        "mtx_right": mtx_right, "dist_right": dist_right,
        "R": R, "T": T, "R1": R1, "R2": R2, "P1": P1, "P2": P2, "Q": Q,
        "image_size": image_size, "square_size": square_size,
        "map_left_x": map_left_x, "map_left_y": map_left_y,
        "map_right_x": map_right_x, "map_right_y": map_right_y,
    }
    return calibration

def get_3dpoints(calibration, sequence_dir, pair_name):
    left, right = read_pair(sequence_dir, pair_name)
    assert left is not None and right is not None, f"Missing left or right image for {pair_name}."
    assert left.shape[:2] == right.shape[:2], "Use the calibration image resolution."
    rectified_left, rectified_right = rectify_pair(left, right, calibration)
    gray_left = cv2.cvtColor(rectified_left, cv2.COLOR_BGR2GRAY)
    gray_right = cv2.cvtColor(rectified_right, cv2.COLOR_BGR2GRAY)

    min_disparity = 6
    num_disparities = 128  # OpenCV requires a multiple of 16
    block_size = 5
    stereo = cv2.StereoSGBM_create(
        minDisparity=min_disparity,
        numDisparities=num_disparities,
        blockSize=block_size,
        P1=2*8 * block_size**2,
        P2=2* 32 * block_size**2,
        disp12MaxDiff=1,
        uniquenessRatio=15,
        speckleWindowSize=150,
        speckleRange=2
    )
    # SGBM returns disparity in sixteenths of a pixel.
    disparity = stereo.compute(gray_left, gray_right).astype(np.float32) / 16
    disparity[disparity < min_disparity] = np.nan


    ################## use WLS filter

    #stereo_right = cv2.ximgproc.createRightMatcher(stereo)
    #wls_filter = cv2.ximgproc.createDisparityWLSFilter(matcher_left=stereo)
#
    #disp_left = stereo.compute(gray_left, gray_right)
    #disp_right = stereo_right.compute(gray_right, gray_left)
#
    #disp_filtered = wls_filter.filter(
    #    disp_left, rectified_left, None, disp_right
    #)
#
    #disparity = disp_filtered.astype(np.float32) / 16
    #disparity[disparity < min_disparity] = np.nan
    #points_3d = cv2.reprojectImageTo3D(disparity, calibration["Q"])

    ##################### End use WLS filter

    points_3d = cv2.reprojectImageTo3D(disparity, calibration["Q"])
    return rectified_left, min_disparity, points_3d

def visulize_pointcloud(points_3d, rectified_left, disparity):
    valid = (
        (disparity > 0) &
        np.isfinite(points_3d).all(axis=2) &
        (points_3d[:, :, 2] > 0)
    )

    xyz = points_3d[valid]
    rgb =(cv2.cvtColor(rectified_left, cv2.COLOR_BGR2RGB)[valid]) / 255.0

    cloud = o3d.geometry.PointCloud()
    cloud.points = o3d.utility.Vector3dVector(xyz)
    cloud.colors = o3d.utility.Vector3dVector(rgb)

    o3d.visualization.draw_geometries([cloud])



Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [3]:
def Create_Stereo():
    min_disparity = 6
    num_disparities = 128  # OpenCV requires a multiple of 16
    block_size = 5
    stereo = cv2.StereoSGBM_create(
        minDisparity=min_disparity,
        numDisparities=num_disparities,
        blockSize=block_size,
        P1=2*8 * block_size**2,
        P2=2* 32 * block_size**2,
        disp12MaxDiff=1,
        uniquenessRatio=15,
        speckleWindowSize=150,
        speckleRange=2
    )
    return stereo

def get_disparity(stereo, calibration, left, right):
    rectified_left, rectified_right = rectify_pair(left, right, calibration)
    gray_left = cv2.cvtColor(rectified_left, cv2.COLOR_BGR2GRAY)
    gray_right = cv2.cvtColor(rectified_right, cv2.COLOR_BGR2GRAY)
    disparity = stereo.compute(gray_left, gray_right).astype(np.float32)  / 16
    disparity[disparity < 3] = 0
    return rectified_left, disparity


In [4]:
project_folder = "C:/Users/niels/OneDrive - Danmarks Tekniske Universitet/9Sem/PFAS/Final_Project/PFAS_FinalProject-main"
calib = calibration(project_folder, save_calib=True)
rect_left, disparity, points = get_3dpoints(calib, Path(project_folder +"/0_calib"), "016.png")
visulize_pointcloud(points, rect_left, disparity )

Usable stereo pairs: 34/37
Right-side corner order reversed in 7 pairs.
Reprojection RMS: left 0.14, right 0.13, stereo 0.32 px
Rectified row difference: median 0.28 px, 95th percentile 0.82 px


In [5]:
import sys

sys.path.append(
    r"C:/Users/niels/OneDrive - Danmarks Tekniske Universitet/9Sem/PFAS/Final_Project/PFAS_FinalProject-main"
)
#from tracking import CustomizedKalmanFilter


In [6]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import glob as gb
import os


def create_backgorund(images):

    # Get 20 evenly spaced frames from the sequence of images
    #background_frames = np.linspace(0, len(images) - 1, 20, dtype=int)

    # Read the selected frames and convert them to grayscale
    gray_frames = []

    for i in images:
        frame = i
        gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray_frames.append(gray_frame)

    # Stack the grayscale frames along a new axis and compute the median across that axis
    stacked_frames = np.stack(gray_frames, axis=0)
    median_frame = np.median(stacked_frames, axis=0).astype(dtype=np.uint8)

    return median_frame


def detect_husky(background, image):

    image_gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Compute the absolute difference between the background and the current image
    diff = cv2.absdiff(background, image_gray)

    # Apply a binary threshold to the difference image to create a mask of the moving objects
    thresh, mask = cv2.threshold(diff, 22, 255, cv2.THRESH_BINARY)

    # Remove the top part of the mask to avoid detecting the sky or other irrelevant areas and apply morphological operations to clean up the mask
    mask[:360, :] = 0
    mask = cv2.erode(mask, None, iterations=1)
    mask = cv2.dilate(mask, None, iterations=7)
    mask = cv2.erode(mask, None, iterations=8)

    # Find contours in the mask and filter them based on area to identify the husky
    cnt = cv2.findContours(mask.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    areas = [cv2.contourArea(c) for c in cnt[0]]
    min_area = 500

    if len(areas) <= 0 or max(areas) < min_area:
        return None
    else:
        robot_cnt = cnt[0][np.argmax(areas)]

    # Compute the bounding box of the detected husky contour
    bounding_box = cv2.boundingRect(robot_cnt)

    return bounding_box

def Draw_boundingbox(image, bounding_box, color):
        if bounding_box is not None:
            x, y, w, h = bounding_box
            if color == 'red':
                cv2.rectangle(image, (x, y), (x + w, y + h), (0, 255, 0), 2)   
            elif color == 'green':
                cv2.circle(img=image, center=(x, y), radius=int((w+h)/4), color=(0, 0, 255))   

                 


In [14]:






pathVideo = Path("C:\\Users\\niels\\OneDrive - Danmarks Tekniske Universitet\\9Sem\\PFAS\\Final_Project\\PFAS_FinalProject-main\\4_circle")
PathVideoString = "C:\\Users\\niels\\OneDrive - Danmarks Tekniske Universitet\\9Sem\\PFAS\\Final_Project\\PFAS_FinalProject-main\\4_circle\\left\\"
all_names = paired_names(pathVideo)
i=0
left_names= []
while i < 20:
    left_names.append(PathVideoString + all_names[i])
    i+=1

    
    
print(left_names)
print("helloWorld")
print(len(all_names))
#left_imgs[]


['C:\\Users\\niels\\OneDrive - Danmarks Tekniske Universitet\\9Sem\\PFAS\\Final_Project\\PFAS_FinalProject-main\\4_circle\\left\\000000.png', 'C:\\Users\\niels\\OneDrive - Danmarks Tekniske Universitet\\9Sem\\PFAS\\Final_Project\\PFAS_FinalProject-main\\4_circle\\left\\000001.png', 'C:\\Users\\niels\\OneDrive - Danmarks Tekniske Universitet\\9Sem\\PFAS\\Final_Project\\PFAS_FinalProject-main\\4_circle\\left\\000002.png', 'C:\\Users\\niels\\OneDrive - Danmarks Tekniske Universitet\\9Sem\\PFAS\\Final_Project\\PFAS_FinalProject-main\\4_circle\\left\\000003.png', 'C:\\Users\\niels\\OneDrive - Danmarks Tekniske Universitet\\9Sem\\PFAS\\Final_Project\\PFAS_FinalProject-main\\4_circle\\left\\000004.png', 'C:\\Users\\niels\\OneDrive - Danmarks Tekniske Universitet\\9Sem\\PFAS\\Final_Project\\PFAS_FinalProject-main\\4_circle\\left\\000005.png', 'C:\\Users\\niels\\OneDrive - Danmarks Tekniske Universitet\\9Sem\\PFAS\\Final_Project\\PFAS_FinalProject-main\\4_circle\\left\\000006.png', 'C:\\Users\\

In [15]:
backgroundimages = all_names[0:20]
stereo = Create_Stereo()
rect_lefts = []
for name in backgroundimages:
    left_img, right_img = read_pair(pathVideo, name)
    rect_left, disparity = get_disparity(stereo= stereo,calibration=calib, left= left_img, right= right_img)
    rect_lefts.append(rect_left)


background = create_backgorund(images=rect_lefts)


In [16]:
names = all_names[0:650]
len(names)


650

In [17]:
from tracking2 import CustomizedKalmanFilter

#kf2D = CustomizedKalmanFilter(dt=1, is3D=False, is1D=False, init_measurement_uncertainty= [0.2, 1], init_uncertainty=40)
#kf1D = CustomizedKalmanFilter(dt=1/30, is3D=False, is1D=True, init_measurement_uncertainty= [0.30])
kf = CustomizedKalmanFilter(dt=1/30, is3D=True, y_init_measurement_uncertainty = 25 )

positions = []
measurements = []
times = []

# Top view map of the trajectory (X to the right, Z away from the camera), map_scale pixels per meter
map_scale = 30
traj = np.zeros((600, 600, 3), dtype=np.uint8)
for meter in range(1, 10):
    cv2.line(traj, (0, 600 - meter * map_scale), (600, 600 - meter * map_scale), (60, 60, 60), 1)
    cv2.putText(traj, "z = {0} m".format(meter), (5, 600 - meter * map_scale - 3), cv2.FONT_HERSHEY_PLAIN, 0.8, (120, 120, 120), 1)
for meter in range(-4, 5):
    cv2.line(traj, (300 + meter * map_scale, 0), (300 + meter * map_scale, 600), (60, 60, 60), 1)
cv2.circle(traj, (300, 595), 5, (255, 255, 255), -1)
cv2.putText(traj, "camera", (310, 595), cv2.FONT_HERSHEY_PLAIN, 1, (255, 255, 255), 1)
output_path = Path("C:\\Users\\niels\\OneDrive - Danmarks Tekniske Universitet\\9Sem\\PFAS\\Final_Project\\PFAS_FinalProject-main\\video4.mp4")
frame_width = traj.shape[1] + rect_lefts[0].shape[1]
frame_height = max(traj.shape[0], rect_lefts[0].shape[0])
video = cv2.VideoWriter(str(output_path), cv2.VideoWriter_fourcc(*'mp4v'), 30,
                        (frame_width, frame_height))
if not video.isOpened():
    raise RuntimeError(f"Could not open video writer for {output_path}")
initialized_3d = False

i=0
for name in names:
    i+=1
    left_img, right_img = read_pair(pathVideo, name)
    rect_left, disparity = get_disparity(stereo= stereo,calibration=calib, left= left_img, right= right_img)
    valid_disp = disparity > 6
    disparity_for_q = np.where(valid_disp, disparity, 0).astype(np.float32)
    points_3d = cv2.reprojectImageTo3D(disparity, calib["Q"])
    bounding_box = detect_husky(background=background, image=rect_left)
    Draw_boundingbox(bounding_box=bounding_box, image= rect_left, color= 'red')
    k = cv2.waitKey(1) & 0xFF
    if k == 27: break
    if bounding_box:
        x, y, w, h = bounding_box
        if(x >1 and y > 1 and x+w > 1 and y+h >1 ):
            xmeas = x+ w
            ymeas = y+ h
            rel_disparity = disparity[y:y+h, x:x+w]
            roi = points_3d[y:y+h, x:x+w]
            roi_valid = valid_disp[y:y+h, x:x+w]
            roi_valid &= np.isfinite(roi).all(axis=2)
            roi_valid &= roi[:, :, 2] > 0
            if np.any(roi):
                print("Gather Valid ROI")
                new_measurement = np.median(roi[roi_valid], axis=0)
                print(f"NewMeasurement : {new_measurement}")
            else:
                new_measurement = None
            #rel_disparity = rel_disparity[rel_disparity != 0]
            #rel_disparity = np.round(rel_disparity)
            #rel_disparity = np.mean(rel_disparity)
            #print(f"relative disparity {rel_disparity}")
            #zmeas = rel_disparity
            #new_measurement = np.array([xmeas, ymeas, zmeas])
            #points_3d = cv2.reprojectImageTo3D(disparity, calib["Q"])
            #kf2D.update_cycle(new_measurement)
            #kf1D.update_cycle(np.array([zmeas]))
        #kf.update_cycle(new_measurement)
    else:
        #kf2D.update_cycle(None)
        #kf1D.update_cycle(None)
        new_measurement = None
    kf.update_cycle(new_measurement)
    Kalman3D = kf.get_x()

    if Kalman3D.all:
        initialized_3d = True

     ##### ################################## #######
    ##### START OF Print and visualize stuff #######
    ##### ##########################################
    print(f"position of robot {Kalman3D}")
   # rect_left = cv2.circle(rect_left, (int(position3D[0]), int(position3D[1])), 60, (0, 255, 255), 2)

    if initialized_3d:
        X, Y, Z = Kalman3D[:3, 0]
        positions.append([X, Y, Z])
        text2 = "No Measurement"
        if new_measurement is not None and np.all(np.isfinite(new_measurement)):
            measurements.append(new_measurement)
            text2 = "Measured Coordinates: x ={0:02f}m y = {1:02f}m z = {2:02f}m".format(float(new_measurement[0]), float(new_measurement[1]), float(new_measurement[2]))

            meas_x, meas_y = int(new_measurement[0] * map_scale) + 300, 600 - int(new_measurement[2] * map_scale)
            cv2.circle(traj, (meas_x, meas_y), 1, (255, 0, 0), 1)
            color = (0, 0, 255)
            status = "Measurement"
        else:
            measurements.append([np.nan, np.nan, np.nan])
            color = (0, 165, 255)
            status = "No measurement - Kalman prediction"
        if np.isfinite(X) and np.isfinite(Y) and np.isfinite(Z):
            draw_x, draw_y = int(X * map_scale) + 300, 600 - int(Z * map_scale)
            cv2.circle(traj, (draw_x, draw_y), 1, color, 2)
    
       #     print(i, X, Y, Z)
    
            text = "Predicted Coordinates: x ={0:02f}m y = {1:02f}m z = {2:02f}m".format(float(X), float(Y), float(Z))
            cv2.rectangle(traj, (10, 30), (590, 70), (0, 0, 0), cv2.FILLED)
            cv2.putText(traj, text, (10, 50), cv2.FONT_HERSHEY_PLAIN, 1, (255, 255, 255), 1, 8)
            cv2.putText(traj, "Frame {0}: {1}".format(i, status), (10, 68), cv2.FONT_HERSHEY_PLAIN, 1, color, 1, 8)
            cv2.putText(traj, text2, (10, 86), cv2.FONT_HERSHEY_PLAIN, 1, (255, 255, 255), 1, 8)
            # 3D estimate projected into the image
            if X and Y and Z:
                projected = (calib["P1"]) @ np.array([X, Y, Z, 1])
                cv2.circle(rect_left, (int(projected[0] / projected[2]), int(projected[1] / projected[2])), 12, color, 3)
    
    h1, w1 = traj.shape[:2]
    h2, w2 = rect_left.shape[:2]
    vis = np.zeros((max(h1, h2), w1 + w2, 3), np.uint8)
    vis[:h1, :w1, :3] = traj
    vis[:h2, w1:w1 + w2, :3] = rect_left

    video.write(vis)
    cv2.imshow("Trajectory", cv2.resize(vis, None, fx=1.2, fy=1.2))
    k = cv2.waitKey(1) & 0xFF
    if k == 27: break

video.release()
cv2.waitKey(0)
cv2.destroyAllWindows()
print('Saved video:', output_path)
#print('Frames with a 3D measurement:', np.count_nonzero(~np.isnan(np.array(measurements)[:, 0])), '/', len(images))
print('Mean time per frame: {0:.1f} ms'.format(np.mean(times) * 1000))
    ##### ################################ #######
    ##### END OF Print and visualize stuff #######
    ##### ########################################
    #position = kf2D.get_x()
    #position1D = kf1D.get_x()
    #if np.all(np.isfinite(position3D)):
    #    filtered_position = np.asarray(position3D).reshape(-1)
    #    cx = int(round(filtered_position[0]))
    #    cy = int(round(filtered_position[1]))
    #    print(f'cx : {cx} ')
    #    print(f'cy : {cy}')
    #    if cy > 10000 or cy < -1000 or cx > 10000 or cx < -1000:
    #        break 
    #    if (cx > 30 and cx < 1000 and cy > 30 and cy < 770):
    #        Draw_boundingbox(
    #            bounding_box=[int(cx), int(cy), 20, 20],
    #            image=rect_left,
    #            color='green'
    #        )
    #if np.all(np.isfinite(position)):
    #    filtered_position = np.asarray(position).reshape(-1)
    #    cx = int(round(filtered_position[0]))
    #    cy = int(round(filtered_position[1]))
    #    print(f'cx : {cx} ')
    #    print(f'cy : {cy}')
    #    if cy > 10000 or cy < -1000 or cx > 10000 or cx < -1000:
    #        break 
    #    if (cx > 30 and cx < 1000 and cy > 30 and cy < 770):
    #        Draw_boundingbox(
    #            bounding_box=[int(cx), int(cy), 20, 20],
    #            image=rect_left,
    #            color='green'
    #        )
    #if np.all(np.isfinite(position3D)):
    #    filtered_position = np.asarray(position3D).reshape(-1)
    #    cz = filtered_position[2]
    #    print(f'cz : {cz}')
    #    if cz > 10000 or cz < -1000:
    #        break 
    #    cv2.putText(
    #        rect_left,
    #        f"Z: {cz:.2f}",
    #        (cx + 10, cy - 10),
    #        cv2.FONT_HERSHEY_SIMPLEX,
    #        0.7,
    #        (0, 0, 255),
    #        2
    #    )
    #    #uncertainty = kf.get_P()
    #rect_left = cv2.resize(rect_left, [1000, 750], interpolation=cv2.INTER_LINEAR)
    #cv2.imshow("lol", rect_left)

#cv2.waitKey(0)
#cv2.destroyAllWindows()

position of robot [[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]
position of robot [[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]
position of robot [[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]
position of robot [[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]
position of robot [[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]
position of robot [[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]
position of robot [[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]
position of robot [[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]
position of robot [[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]
position of robot [[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]
position of robot [[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]
position of robot [[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]
position of robot [[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]
position of robot [[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0

C:\Users\niels\miniconda3\envs\pfas\lib\site-packages\numpy\_core\_methods.py:130: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


Gather Valid ROI
NewMeasurement : [nan nan nan]
position of robot [[nan]
 [nan]
 [nan]
 [nan]
 [nan]
 [nan]
 [nan]
 [nan]
 [nan]]
Gather Valid ROI
NewMeasurement : [ 2.245503   1.0674413 12.681904 ]
position of robot [[nan]
 [nan]
 [nan]
 [nan]
 [nan]
 [nan]
 [nan]
 [nan]
 [nan]]
Gather Valid ROI
NewMeasurement : [nan nan nan]
position of robot [[nan]
 [nan]
 [nan]
 [nan]
 [nan]
 [nan]
 [nan]
 [nan]
 [nan]]
Gather Valid ROI
NewMeasurement : [nan nan nan]
position of robot [[nan]
 [nan]
 [nan]
 [nan]
 [nan]
 [nan]
 [nan]
 [nan]
 [nan]]
Gather Valid ROI
NewMeasurement : [nan nan nan]
position of robot [[nan]
 [nan]
 [nan]
 [nan]
 [nan]
 [nan]
 [nan]
 [nan]
 [nan]]
Gather Valid ROI
NewMeasurement : [nan nan nan]
position of robot [[nan]
 [nan]
 [nan]
 [nan]
 [nan]
 [nan]
 [nan]
 [nan]
 [nan]]
Gather Valid ROI
NewMeasurement : [nan nan nan]
position of robot [[nan]
 [nan]
 [nan]
 [nan]
 [nan]
 [nan]
 [nan]
 [nan]
 [nan]]
Gather Valid ROI
NewMeasurement : [nan nan nan]
position of robot [[n